# SLIS — Step 4: Run API Server & Test Endpoints

This notebook:
1. Starts the FastAPI server in a background thread
2. Tests all endpoints with `requests`
3. Shows recommendations powered by **HuggingFace Qwen3-32B**

> **Run notebooks 01 and 03 first** (data + models must exist).

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os, time, json, threading
import requests
import uvicorn

# Load HF token
hf_token_file = Path.home() / '.hf_token'
if hf_token_file.exists():
    os.environ['HF_TOKEN'] = hf_token_file.read_text().strip()
    print('HF_TOKEN loaded from ~/.hf_token')
elif os.environ.get('HF_TOKEN'):
    print('HF_TOKEN already in environment')
else:
    print('WARNING: HF_TOKEN not set — recommendations will use rule-based fallback')

BASE_URL = 'http://127.0.0.1:8000'
print('Base URL:', BASE_URL)

## Start FastAPI Server (background thread)

In [ ]:
from backend.main import app

config = uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning')
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

# Wait for server to be ready
for _ in range(20):
    try:
        r = requests.get(f'{BASE_URL}/health', timeout=1)
        if r.status_code == 200:
            print('Server ready:', r.json())
            break
    except:
        time.sleep(0.5)
else:
    print('Server did not start in time')

## Health Check

In [ ]:
r = requests.get(f'{BASE_URL}/health')
print(json.dumps(r.json(), indent=2))

## List Students (first page)

In [ ]:
import pandas as pd
r = requests.get(f'{BASE_URL}/api/students?page=1&limit=10')
data = r.json()
print(f'Total students: {data["total"]}')
pd.DataFrame(data['students'])

## Filter High-Risk Students

In [ ]:
r = requests.get(f'{BASE_URL}/api/students?risk_filter=High&limit=10')
data = r.json()
print(f'High-risk students: {data["total"]}')
pd.DataFrame(data['students'])[['student_id','name','major','avg_score','avg_attendance','risk_level']]

## Get Single Student Detail

In [ ]:
student_id = 'STU0001'
r = requests.get(f'{BASE_URL}/api/students/{student_id}')
s = r.json()
print(f"Name: {s['name']} | Major: {s['major']}")
print(f"Attendance: {s['avg_attendance']}% | Score: {s['avg_score']} | Risk: {s['risk_level']}")
print(f"Risk probabilities: {s['risk_probabilities']}")
print(f"Predicted score: {s['predicted_score']}")
print('\nScores by subject:')
pd.DataFrame(s['scores_by_subject'])

## Predict Risk (custom input)

In [ ]:
r = requests.post(f'{BASE_URL}/api/predict', json={
    'avg_attendance': 62.0,
    'engagement_score': 4.2,
    'gpa_start': 5.8,
    'lms_logins_per_week': 3.5,
    'forum_posts': 2.0,
})
print(json.dumps(r.json(), indent=2))

## Dashboard Summary

In [ ]:
r = requests.get(f'{BASE_URL}/api/dashboard')
data = r.json()
print(json.dumps(data, indent=2))

## AI Recommendations (HuggingFace Qwen3-32B)
Calls the `/api/recommendations/{student_id}` endpoint which uses Qwen3-32B via HF Inference API.

In [ ]:
student_id = 'STU0001'
r = requests.get(f'{BASE_URL}/api/recommendations/{student_id}')
data = r.json()
print(f"Recommendations for {data['student_name']}\n" + '='*60)
for i, rec in enumerate(data['recommendations'], 1):
    print(f"\n{i}. [{rec['priority']}] {rec['title']}")
    print(f"   {rec['description']}")

## Test a High-Risk Student's Recommendations

In [ ]:
# Find a high-risk student
r = requests.get(f'{BASE_URL}/api/students?risk_filter=High&limit=1')
high_risk_id = r.json()['students'][0]['student_id'] if r.json()['students'] else 'STU0001'
print(f'Testing high-risk student: {high_risk_id}')

r = requests.get(f'{BASE_URL}/api/recommendations/{high_risk_id}')
data = r.json()
print(f"\nRecommendations for {data['student_name']}\n" + '='*60)
for i, rec in enumerate(data['recommendations'], 1):
    print(f"\n{i}. [{rec['priority']}] {rec['title']}")
    print(f"   {rec['description']}")

## API Docs

With the server running above, open: http://127.0.0.1:8000/docs

This gives you the full interactive Swagger UI for all endpoints.

In [ ]:
print('Swagger UI: http://127.0.0.1:8000/docs')
print('ReDoc     : http://127.0.0.1:8000/redoc')
print('\nAll endpoints:')
r = requests.get(f'{BASE_URL}/openapi.json')
for path in r.json()['paths']:
    print(' ', path)